# EDIT / ESD MLRA Catalog Extractor

This notebook builds the first dataset for the ESD database: a table of Major Land Resource Areas (MLRAs) from the EDIT ESD catalog. It is designed to work from either a saved local HTML file or a live EDIT URL.

Outputs:
- `outputs/mlra_catalog.csv`
- `outputs/mlra_catalog.sqlite`

## 0. Setup

In [6]:
from pathlib import Path
import re
import sqlite3
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://edit.sc.egov.usda.gov"
CATALOG_URL = f"{BASE_URL}/catalogs/esd"

# Change this path if needed. Use None to fetch from the web.
LOCAL_CATALOG_HTML = Path("/mnt/data/referenceESD.html")

OUT_DIR = Path("C:/NCA_DATA/Ancillary_Data/NRCS ESDs")
OUT_DIR.mkdir(exist_ok=True)

## 1. Load catalog HTML

Use the saved catalog page first. Later, set `LOCAL_CATALOG_HTML = None` to fetch a fresh copy from EDIT.

In [7]:
def load_html(local_path=None, url=CATALOG_URL, timeout=60):
    if local_path is not None and Path(local_path).exists():
        html = Path(local_path).read_text(encoding="utf-8", errors="replace")
        source = str(local_path)
    else:
        headers = {"User-Agent": "Mozilla/5.0 (compatible; ESD research data extraction)"}
        r = requests.get(url, headers=headers, timeout=timeout)
        r.raise_for_status()
        html = r.text
        source = url
    return html, source

html, html_source = load_html(LOCAL_CATALOG_HTML)
print(f"Loaded HTML from: {html_source}")
print(f"Characters: {len(html):,}")

Loaded HTML from: https://edit.sc.egov.usda.gov/catalogs/esd
Characters: 944,656


## 2. Parse MLRA catalog cards

The catalog page contains one `<li class="content-navigation-list-item">` per MLRA. Each card has the MLRA symbol, name, link, internal unit id, and ecological-site count.

In [8]:
def parse_mlra_catalog(html, base_url=BASE_URL):
    soup = BeautifulSoup(html, "html.parser")
    records = []

    for li in soup.select("li.content-navigation-list-item"):
        symbol_el = li.select_one(".list-item-symbol")
        name_el = li.select_one(".list-item-name")
        title_a = li.select_one("a.list-item-title")
        count_el = li.select_one(".list-item-class-count")

        if not symbol_el or not name_el:
            continue

        count = None
        if count_el:
            m = re.search(r"(\d+)", count_el.get_text(" ", strip=True))
            count = int(m.group(1)) if m else None

        href = title_a.get("href") if title_a else None
        records.append({
            "mlra_symbol": symbol_el.get_text(" ", strip=True),
            "mlra_name": name_el.get_text(" ", strip=True),
            "edit_unit_id": li.get("unit"),
            "mlra_url": urljoin(base_url, href) if href else None,
            "ecological_site_count": count,
            "hidden_until_view_all": "overflow" in (li.get("class") or []),
        })

    df = pd.DataFrame.from_records(records)
    if not df.empty:
        df = df.sort_values("mlra_symbol").reset_index(drop=True)
    return df

mlra_df = parse_mlra_catalog(html)
print(mlra_df.shape)
mlra_df.head(10)

(267, 6)


,mlra_symbol,mlra_name,edit_unit_id,mlra_url,ecological_site_count,hidden_until_view_all
0,001X,"Northern Pacific Coast Range, Foothills, and V...",27,https://edit.sc.egov.usda.gov/catalogs/esd/001X,53,False
1,002X,Willamette and Puget Sound Valleys,28,https://edit.sc.egov.usda.gov/catalogs/esd/002X,43,False
2,003X,Olympic and Cascade Mountains,29,https://edit.sc.egov.usda.gov/catalogs/esd/003X,79,False
3,004A,Sitka Spruce Belt,30,https://edit.sc.egov.usda.gov/catalogs/esd/004A,31,False
4,004B,Coastal Redwood Belt,31,https://edit.sc.egov.usda.gov/catalogs/esd/004B,65,False
5,005X,Siskiyou-Trinity Area,33,https://edit.sc.egov.usda.gov/catalogs/esd/005X,2,False
6,006X,"Cascade Mountains, Eastern Slope",34,https://edit.sc.egov.usda.gov/catalogs/esd/006X,79,False
7,007X,Columbia Basin,35,https://edit.sc.egov.usda.gov/catalogs/esd/007X,23,False
8,008X,Columbia Plateau,36,https://edit.sc.egov.usda.gov/catalogs/esd/008X,36,False
9,009X,Palouse and Nez Perce Prairies,37,https://edit.sc.egov.usda.gov/catalogs/esd/009X,83,False


## 3. Basic validation

In [9]:
assert not mlra_df.empty, "No MLRA records parsed. Check source HTML."
assert mlra_df["mlra_symbol"].is_unique, "MLRA symbols are not unique."

print("MLRAs:", len(mlra_df))
print("Total ecological site count from catalog:", int(mlra_df["ecological_site_count"].fillna(0).sum()))
print("Missing site counts:", mlra_df["ecological_site_count"].isna().sum())
mlra_df.describe(include="all")

MLRAs: 267
Total ecological site count from catalog: 8300
Missing site counts: 0


,mlra_symbol,mlra_name,edit_unit_id,mlra_url,ecological_site_count,hidden_until_view_all
count,267,267,267,267,267.000000,267
unique,267,267,267,267,NaN,2
top,001X,"Northern Pacific Coast Range, Foothills, and V...",27,https://edit.sc.egov.usda.gov/catalogs/esd/001X,NaN,True
freq,1,1,1,1,NaN,247
mean,NaN,NaN,NaN,NaN,31.086142,NaN
std,NaN,NaN,NaN,NaN,47.828839,NaN
min,NaN,NaN,NaN,NaN,0.000000,NaN
25%,NaN,NaN,NaN,NaN,9.000000,NaN
50%,NaN,NaN,NaN,NaN,18.000000,NaN
75%,NaN,NaN,NaN,NaN,32.000000,NaN


## 4. Save MLRA dataset

In [10]:
csv_path = OUT_DIR / "mlra_catalog.csv"
sqlite_path = OUT_DIR / "mlra_catalog.sqlite"

mlra_df.to_csv(csv_path, index=False)

with sqlite3.connect(sqlite_path) as con:
    mlra_df.to_sql("mlra_catalog", con, if_exists="replace", index=False)

print(csv_path.resolve())
print(sqlite_path.resolve())

C:\NCA_DATA\Ancillary_Data\NRCS ESDs\mlra_catalog.csv
C:\NCA_DATA\Ancillary_Data\NRCS ESDs\mlra_catalog.sqlite


## 5. Optional: parse ecological-site links from one MLRA page

This is the next layer after the MLRA catalog. It will let the database move from `MLRA -> site list -> site HTML -> structured ESD tables`. Run this after saving an individual MLRA page or enabling web fetches.

In [ ]:
def fetch_html(url, sleep_seconds=0.5, timeout=60):
    headers = {"User-Agent": "Mozilla/5.0 (compatible; ESD research data extraction)"}
    r = requests.get(url, headers=headers, timeout=timeout)
    r.raise_for_status()
    time.sleep(sleep_seconds)
    return r.text

def parse_ecosite_links_from_mlra_page(html, mlra_symbol, base_url=BASE_URL):
    soup = BeautifulSoup(html, "html.parser")
    records = []

    # Site pages generally have links containing /catalogs/esd/{MLRA}/R...
    pattern = re.compile(rf"/catalogs/esd/{re.escape(mlra_symbol)}/[A-Z0-9]+", re.I)

    seen = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if not pattern.search(href):
            continue
        url = urljoin(base_url, href)
        ecoclassid = url.rstrip("/").split("/")[-1]
        if ecoclassid in seen:
            continue
        seen.add(ecoclassid)
        text = a.get_text(" ", strip=True)
        records.append({
            "mlra_symbol": mlra_symbol,
            "ecoclassid": ecoclassid,
            "site_url": url,
            "link_text": text,
        })

    return pd.DataFrame.from_records(records)

# Example use with web fetch enabled:
# row = mlra_df.loc[mlra_df["mlra_symbol"].eq("011X")].iloc[0]
# mlra_html = fetch_html(row["mlra_url"])
# sites_011x = parse_ecosite_links_from_mlra_page(mlra_html, "011X")
# sites_011x.head()

## 6. Optional: filter western target domain

State filtering should not be hard-coded from MLRA names. The better route is to use the MLRA GeoJSON exposed in the catalog page, spatially intersect it with target-state polygons, then retain MLRAs intersecting WA, OR, CA, ID, NV, AZ, UT, MT, WY, NM, and CO. That is a separate spatial enrichment step.

For now, this notebook produces the complete MLRA catalog. The next notebook can add the spatial state/MLRA intersection.

In [ ]:
TARGET_STATES = ["WA", "OR", "CA", "ID", "NV", "AZ", "UT", "MT", "WY", "NM", "CO"]
print("Target states for later spatial filtering:", TARGET_STATES)